In [0]:
-- 03_gold_03_actions
-- Layer: gold (actionable intelligence)
-- Purpose: implement the 5 whys + 1 how framework for root cause analysis (rca).
-- Method: cascading diagnostic tables with pre-calculated "why" justifications

-- 1. Business health (executive summary)
-- Example question: "how is the business performing across core pillars?"

CREATE OR REPLACE TABLE olist_maas_pipeline.gold.report_business_vitals
USING DELTA
AS
SELECT 
    DATE_TRUNC('month', purchase_date) AS reporting_month
    , ROUND(SUM(total_payment), 2) AS total_gmv
    , ROUND(AVG(actual_delivery_days), 1) AS avg_delivery_time
    , ROUND(COUNT(CASE WHEN order_status = 'canceled' THEN 1 END) * 100.0 / COUNT(*), 2) AS cancellation_rate
    -- health status logic
    , CASE 
        WHEN AVG(actual_delivery_days) > 12 THEN 'UNHEALTHY: logistics bottleneck'
        WHEN COUNT(CASE WHEN order_status = 'canceled' THEN 1 END) * 100.0 / COUNT(*) > 5 THEN 'UNHEALTHY: high churn at checkout'
        ELSE 'HEALTHY' 
      END AS business_status
FROM olist_maas_pipeline.gold.fact_orders
GROUP BY 1;

-- 2. Revenue protection (the 5 whys)
-- Example question: "why are our highest spending customers leaving, and how do we save them?"

CREATE OR REPLACE TABLE olist_maas_pipeline.gold.rca_customer_churn_5whys
USING DELTA
AS
SELECT 
    customer_unique_id
    , total_spend AS revenue_at_risk
    , 'REVENUE_LOSS' AS why_1_symptom
    , customer_state AS why_2_location
    , CASE 
        WHEN extreme_delay_orders > 0 THEN 'BREAKING_POINT_REACHED'
        ELSE 'PRICE_SENSITIVITY'
      END AS why_3_behavior
    , 'LOGISTICS_LATENCY' AS why_4_operational
    , 'TIER_3_INFRASTRUCTURE_FAILURE' AS why_5_root_cause
    , 'RE-ROUTE_THROUGH_TIER_1_HUB' AS the_1_how_remedy
FROM olist_maas_pipeline.gold.dim_customer
WHERE rfm_segment = 'CANT_LOSE_THEM' AND churn_probability_score > 70;

-- 3.Operational leakage (rca)
-- Example question: "why are we losing money on cancellations in specific regions?"

CREATE OR REPLACE TABLE olist_maas_pipeline.gold.rca_cancellation_leakage
USING DELTA
AS
SELECT 
    o.region_name
    , COUNT(o.order_id) AS canceled_volume
    , SUM(p.total_payment) AS leaked_revenue
    , 'REVENUE_LEAKAGE' AS why_1_symptom
    , o.region_name AS why_2_location
    , 'ORDER_ABANDONMENT' AS why_3_behavior
    , 'LONG_LEAD_TIMES' AS why_4_operational
    , 'INACCURATE_DELIVERY_ESTIMATES' AS why_5_root_cause
    , 'ADJUST_DYNAMIC_ESTIMATION_ALGORITHM' AS the_1_how_remedy
FROM olist_maas_pipeline.gold.fact_orders o
JOIN olist_maas_pipeline.gold.int_payments_summary p ON o.order_id = p.order_id
WHERE o.order_status = 'canceled'
GROUP BY 1;

-- 4. Product innovation & early adopters
-- Example question: "who are our early adopters for new features, and why?"

CREATE OR REPLACE TABLE olist_maas_pipeline.gold.action_early_adopter_leads
USING DELTA
AS
SELECT 
    customer_unique_id
    , 'ELITE_PROMOTER' AS adoption_propensity
    , 'HIGH_BRAND_TRUST_AND_RECENCY' AS adoption_why
    , 'INVITE_TO_BETA_TEST_PROGRAM' AS the_1_how_remedy
FROM olist_maas_pipeline.gold.dim_customer
WHERE rfm_segment = 'CHAMPIONS' AND avg_rating >= 4.5;

-- 5. Logistics diagnostic gate
-- Example question: "is the current sla breach an external infrastructure crisis or an internal merchant failure?"
CREATE OR REPLACE TABLE olist_maas_pipeline.gold.action_seller_intervention
USING DELTA
AS
WITH regional_benchmark AS (
    -- establishing the "neighborhood" baseline using silver tier
    SELECT 
        s.seller_region
        , l.logistics_tier
        , l.breaking_point_days AS regional_limit
        , AVG(s.on_time_rate) AS regional_sla_baseline
    FROM olist_maas_pipeline.gold.dim_seller s
    JOIN olist_maas_pipeline.silver.dim_logistics_regions l ON s.seller_state = l.state_code
    GROUP BY 1, 2, 3
)
SELECT 
    s.seller_id
    , s.seller_region
    , r.logistics_tier
    , s.on_time_rate AS merchant_performance
    , r.regional_sla_baseline AS market_performance
    , r.regional_limit AS tier_breaking_point
    -- the 5 whys navigation
    , 'SLA_THRESHOLD_BREACH' AS why_1_symptom
    , CONCAT(s.seller_region, ' (Tier ', r.logistics_tier, ')') AS why_2_location
    -- why 3: diagnosis is now sensitive to the tier's infrastructure
    , CASE 
        WHEN (r.regional_sla_baseline - s.on_time_rate) > 0.20 THEN 'MERCHANT_SPECIFIC_OUTLIER'
        WHEN r.logistics_tier = 3 AND s.on_time_rate < 0.65 THEN 'REGIONAL_INFRASTRUCTURE_STRESS'
        WHEN r.regional_sla_baseline < 0.70 THEN 'SYSTEMIC_LOGISTICS_CRISIS'
        ELSE 'STANDARD_OPERATIONAL_DRIFT'
      END AS why_3_diagnosis
    -- why 5: the root cause
    , CASE 
        WHEN (r.regional_sla_baseline - s.on_time_rate) > 0.20 THEN 'INTERNAL_FULFILLMENT_DEFICIT'
        WHEN r.logistics_tier = 3 THEN 'SPARSE_INFRASTRUCTURE_FRICTION'
        ELSE 'EXTERNAL_SUPPLY_CHAIN_BOTTLENECK'
      END AS why_5_root_cause
    -- the 1 how: strategic action (the business result)
    , CASE 
        WHEN (r.regional_sla_baseline - s.on_time_rate) > 0.20 THEN 'INITIATE_MERCHANT_CORRECTIVE_ACTION_PLAN'
        WHEN r.logistics_tier = 3 THEN 'ADJUST_CUSTOMER_EXPECTATIONS: regional buffer enabled'
        ELSE 'ACTIVATE_SLA_PROTECTION_PROTOCOL'
      END AS the_1_how_action
FROM olist_maas_pipeline.gold.dim_seller s
JOIN regional_benchmark r ON s.seller_region = r.seller_region
WHERE s.logistics_reliability_tier = 'LOGISTICS_CRISIS';
